### Install KerasTuner
pip install keras-tuner

# Hyperparameter Tuning using Hyperband (KerasTuner)

In [ ]:
# ============================================================
# Hyperparameter Tuning using Hyperband (KerasTuner)
# ============================================================

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Load Dataset
# -----------------------------
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("Training Images :", X_train.shape)
print("Testing Images  :", X_test.shape)

# -----------------------------
# Display Sample Images
# -----------------------------
plt.figure(figsize=(10,6))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(X_train[i], cmap="gray")

    plt.title(y_train[i])

    plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# Normalize Images
# -----------------------------
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train.reshape(-1,784)
X_test = X_test.reshape(-1,784)

# -----------------------------
# Build HyperModel
# -----------------------------
def build_model(hp):

    model = keras.Sequential()

    model.add(
        keras.layers.Input(shape=(784,))
    )

    units = hp.Int(
        "units",
        min_value=64,
        max_value=256,
        step=64
    )

    model.add(
        keras.layers.Dense(
            units,
            activation="relu"
        )
    )

    dropout = hp.Float(
        "dropout",
        min_value=0.2,
        max_value=0.5,
        step=0.1
    )

    model.add(
        keras.layers.Dropout(dropout)
    )

    model.add(
        keras.layers.Dense(
            128,
            activation="relu"
        )
    )

    learning_rate = hp.Choice(
        "learning_rate",
        values=[0.01,0.001,0.0001]
    )

    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )

    model.add(
        keras.layers.Dense(
            10,
            activation="softmax"
        )
    )

    model.compile(

        optimizer=optimizer,

        loss="sparse_categorical_crossentropy",

        metrics=["accuracy"]

    )

    return model

# -----------------------------
# Hyperband Tuner
# -----------------------------
tuner = kt.Hyperband(

    build_model,

    objective="val_accuracy",

    max_epochs=10,

    factor=3,

    directory="hyperband",

    project_name="mnist"

)

# -----------------------------
# Early Stopping Callback
# -----------------------------
stop_early = keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3

)

# -----------------------------
# Search Best Hyperparameters
# -----------------------------
tuner.search(

    X_train,

    y_train,

    epochs=10,

    validation_split=0.2,

    callbacks=[stop_early],

    verbose=1

)

# -----------------------------
# Best Hyperparameters
# -----------------------------
best_hp = tuner.get_best_hyperparameters(1)[0]

print("\nBest Hyperparameters")

print("Units         :", best_hp.get("units"))
print("Dropout       :", best_hp.get("dropout"))
print("Learning Rate :", best_hp.get("learning_rate"))

# -----------------------------
# Build Best Model
# -----------------------------
model = tuner.hypermodel.build(best_hp)

# -----------------------------
# Train Best Model
# -----------------------------
history = model.fit(

    X_train,

    y_train,

    epochs=10,

    validation_split=0.2,

    verbose=1

)

# -----------------------------
# Evaluate Model
# -----------------------------
test_loss, test_accuracy = model.evaluate(

    X_test,

    y_test

)

print("\nTest Accuracy :", test_accuracy)

# -----------------------------
# Plot Accuracy
# -----------------------------
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)

plt.plot(
    history.history["accuracy"],
    label="Training"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()

# -----------------------------
# Plot Loss
# -----------------------------
plt.subplot(1,2,2)

plt.plot(
    history.history["loss"],
    label="Training"
)

plt.plot(
    history.history["val_loss"],
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()

plt.tight_layout()
plt.show()

# -----------------------------
# Predictions
# -----------------------------
predictions = model.predict(X_test)

predicted_labels = np.argmax(predictions, axis=1)

# -----------------------------
# Display Sample Predictions
# -----------------------------
plt.figure(figsize=(10,6))

for i in range(12):

    plt.subplot(3,4,i+1)

    plt.imshow(
        X_test[i].reshape(28,28),
        cmap="gray"
    )

    color = "green" if predicted_labels[i] == y_test[i] else "red"

    plt.title(
        f"P:{predicted_labels[i]}\nT:{y_test[i]}",
        color=color
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

# -----------------------------
# Single Prediction
# -----------------------------
index = 100

prediction = np.argmax(

    model.predict(

        X_test[index].reshape(1,784)

    )

)

print("Actual Label    :", y_test[index])
print("Predicted Label :", prediction)